### <font color=Blue><font color=Blue>This Code demonstartes how to pass the response of one prompt as input to next prompt using simple sequential prompting.

In [2]:
from getpass import getpass
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import SimpleSequentialChain
from langchain_core.prompts import FewShotPromptTemplate
import os

#### Create LLM Instance

In [3]:
from langchain_aws import ChatBedrockConverse
llm1=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm1.invoke("Hi").content

'Hello! How can I help you today?'

<font color=Blue><b> Chain1 : An LLM which generate a brief note about Bird

In [4]:
template1 = "You are a writer. You are Given with the name of a Bird, it is your job to write a brief note for that Name.\
             Name: {name}   BriefNote: This is a briefnote for the above bird:"
prompt1 = PromptTemplate(template=template1, input_variables=["name"])
Chain_one = prompt1 | llm1

<font color=Blue><b>Chain2 :An LLM which review the brief note generated by considering previous prompt

In [5]:
from langchain_aws import ChatBedrockConverse
llm2=ChatBedrockConverse(model='amazon.nova-lite-v1:0',
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm2.invoke("Hi").content


"Hello! How can I assist you today? If you have any questions or need information on a particular topic, feel free to ask. Whether it's about science, technology, history, or something else, I'm here to help."

In [6]:

template2 = "You are an expert in zoology. Given a brief note about a bird, it is your job to write a review comments for that Bird.\
             briefNote: {note}   Review: Review for a briefnote on bird: "
prompt2 = PromptTemplate(template=template2, input_variables=["note"])
Chain_two = prompt2 | llm2 

In [11]:
#Use if next code doesn't work
#Chain_one.invoke("Parrot")['text']

In [12]:
note=Chain_one.invoke("peacock").content

print(note)

"The peacock, with its exquisite plumage and vibrant hues, is a true embodiment of natural beauty and grace. Its elaborate tail feathers, adorned with eyespots of iridescent blues, greens, and golds, serve not only as a stunning visual display but also as a symbol of pride and confidence. This majestic bird, known for its elegant strut and distinctive call, captivates onlookers with its mesmerizing dances, leaving them in awe of nature's wondrous creations."


In [13]:
print(Chain_two.invoke(note).content)

**Review for the Brief Note on the Peacock:**

The brief note on the peacock is a poetic and vivid encapsulation of one of nature's most magnificent avian wonders. The peacock, with its dazzling plumage and vibrant, multifaceted hues, truly stands as an emblem of natural beauty and elegance. The author aptly highlights the bird's elaborate tail feathers, which are not only a visual spectacle but also rich with symbolic meaning.

The description of the peacock's tail feathers, adorned with their captivating eyespots of iridescent blues, greens, and golds, is particularly evocative. These feathers are more than just a feast for the eyes; they are a testament to the artistry of evolution. The mention of the peacock's symbol of pride and confidence adds a layer of depth, connecting the bird's physical attributes with its behavioral and symbolic significance.

Furthermore, the note eloquently captures the peacock's demeanor and behavior. The elegant strut and distinctive call of the peacock

## Running as a Simple Sequential Chain with Combined Chain

In [16]:
from langchain_core.runnables import RunnableLambda
from rich import print

# Define a logging step to show the intermediate output
log_note = RunnableLambda(lambda x: print("📝 Note on bird:", x) or x)  # Logs intermediate result and Passes the same data to the next chain step

# Now combine the full chain
combined_chain = Chain_one | log_note | Chain_two

# Invoke the full chain
final_output = combined_chain.invoke({"name": "bird of paradise"})

# Show final result

print("___"*45,"✅ Final output:", final_output)


📝 Note on bird:
AIMessage(
    content='The bird of paradise is an exquisite creature, a true masterpiece of evolution. With vibrant plumage 
that seems to defy the very laws of nature, it is no wonder that this bird has captured the imagination of people 
for centuries. The intricate dance rituals and the stunning display of colors make this bird an embodiment of 
natural beauty and grace. A true paradise indeed!',
    additional_kwargs={},
    response_metadata={
        'ResponseMetadata': {
            'RequestId': '39f43a45-ef13-49eb-8c7b-85a287a90793',
            'HTTPStatusCode': 200,
            'HTTPHeaders': {
                'date': 'Mon, 16 Mar 2026 06:50:48 GMT',
                'content-type': 'application/json',
                'content-length': '578',
                'connection': 'keep-alive',
                'x-amzn-requestid': '39f43a45-ef13-49eb-8c7b-85a287a90793'
            },
            'RetryAttempts': 0
        },
        'stopReason': 'end_turn',
        'metrics': {'latencyMs': [1888]},
        'model_provider': 'bedrock_converse',
        'model_name': 'cohere.command-r-plus-v1:0'
    },
    id='lc_run--019cf569-5411-7b41-891d-e6724aa31379-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 48,
        'output_tokens': 70,
        'total_tokens': 118,
        'input_token_details': {'cache_creation': 0, 'cache_read': 0}
    }
)

___________________________________________________________________________________________________________________
____________________ ✅ Final output:
AIMessage(
    content="### Review: A Masterpiece of Evolution\n\nThe Bird of Paradise is truly a marvel of nature, a living 
testament to the beauty and complexity of evolutionary processes. Its vibrant and seemingly otherworldly plumage 
captivates the eye and ignites the imagination, making it one of the most stunning avian species on the planet. 
\n\nOne of the most remarkable aspects of the Bird of Paradise is its intricate and elaborate dance rituals. These 
performances are not only a spectacle of natural grace but also serve an essential role in the bird's mating 
rituals. The male's stunning display of colors and movements is designed to attract a mate, showcasing the 
evolutionary mastery at play.\n\nFor centuries, the Bird of Paradise has inspired awe and wonder among people, from
naturalists to artists and enthusiasts. Its beauty and elegance are unparalleled, and its presence in the wild is a
true privilege. \n\nIn summary, the Bird of Paradise is more than just a bird; it is a living masterpiece that 
embodies the essence of natural beauty and evolutionary perfection.",
    additional_kwargs={},
    response_metadata={
        'ResponseMetadata': {
            'RequestId': 'e7d1d4e5-886b-44a2-98e1-592846ec8526',
            'HTTPStatusCode': 200,
            'HTTPHeaders': {
                'date': 'Mon, 16 Mar 2026 06:50:50 GMT',
                'content-type': 'application/json',
                'content-length': '1275',
                'connection': 'keep-alive',
                'x-amzn-requestid': 'e7d1d4e5-886b-44a2-98e1-592846ec8526'
            },
            'RetryAttempts': 0
        },
        'stopReason': 'max_tokens',
        'metrics': {'latencyMs': [1274]},
        'model_provider': 'bedrock_converse',
        'model_name': 'amazon.nova-lite-v1:0'
    },
    id='lc_run--019cf569-5c40-7492-ac0c-6f17f269579c-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 478,
        'output_tokens': 200,
        'total_tokens': 678,
        'input_token_details': {'cache_creation': 0, 'cache_read': 0}
    }
)